In [1]:
import os
import sys

from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader, TextLoader

pdf_files=[
    "Data/Data-Structures-in-Python.pdf",
    "Data/OOP-Workbook.pdf",
    "Data/Python Crash Course.pdf"
]

html_file=["Data/Data Types & Operators.html"]

text_files=[
    "Data/Flow-of-control-in-Python.txt",
    "Data/Python-Functions.txt"
]

All_Documents=[]
total_characters_from_all_files=0

def load_file_and_extend(file_list, loader_class, file_type):
    global All_Documents
    global total_characters_from_all_files
    
    print(f"\n ------- loading {file_type} files ---------")
    for file_path in file_list:
        
        if not os.path.exists(file_path):
            print(f"Error : {file_type} file '{file_path}' not found!!")
            
        loader=loader_class(file_path)
        documents_from_current_files=loader.load()
        All_Documents.extend(documents_from_current_files)
        
        current_file_chars=sum(len(doc.page_content) for doc in documents_from_current_files)
        total_characters_from_all_files=total_characters_from_all_files+current_file_chars
        
        print(f"loaded {len(documents_from_current_files)} documents from '{file_path}'")
        print(f"Total Character from the current file : {current_file_chars:,}")
        
load_file_and_extend(pdf_files, PyPDFLoader, "PDF")
load_file_and_extend(text_files, lambda path: TextLoader(path, encoding='cp1252'), "TXT")
load_file_and_extend(html_file, UnstructuredHTMLLoader, "HTML")

print(f"\n✓ Successfully loaded a total of {len(All_Documents)} pages/documents from all files.")
print(f"  Grand total characters across all files: {total_characters_from_all_files:,}")
            



 ------- loading PDF files ---------
loaded 23 documents from 'Data/Data-Structures-in-Python.pdf'
Total Character from the current file : 42,212
loaded 40 documents from 'Data/OOP-Workbook.pdf'
Total Character from the current file : 68,045
loaded 562 documents from 'Data/Python Crash Course.pdf'
Total Character from the current file : 1,048,994

 ------- loading TXT files ---------
loaded 1 documents from 'Data/Flow-of-control-in-Python.txt'
Total Character from the current file : 4,238
loaded 1 documents from 'Data/Python-Functions.txt'
Total Character from the current file : 16,289

 ------- loading HTML files ---------
loaded 1 documents from 'Data/Data Types & Operators.html'
Total Character from the current file : 36,268

✓ Successfully loaded a total of 628 pages/documents from all files.
  Grand total characters across all files: 1,216,046


In [2]:

# SPLIT DOCUMENTS INTO CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks=splitter.split_documents(All_Documents)

print(f"Split into {len(chunks)} chunks\n")



Split into 1690 chunks



In [3]:
from dotenv import load_dotenv

load_dotenv()

if os.getenv("GOOGLE_API_KEY"):
    print("✅ GOOGLE_API_KEY found")
else:
    print("❌ GOOGLE_API_KEY not found")
    print("   Create a .env file with: GOOGLE_API_KEY=your-key-here")

✅ GOOGLE_API_KEY found


In [4]:
# Create Embeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings=GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)
embeddings

GoogleGenerativeAIEmbeddings(client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x12647c9e0>, async_client=None, model='models/text-embedding-004', task_type=None, google_api_key=SecretStr('**********'), credentials=None, client_options=None, base_url=None, transport=None, request_options=None)

In [5]:
# Create Vector Store

from langchain_community.vectorstores import Chroma

vector_store=Chroma.from_documents(
    documents=All_Documents,
    embedding=embeddings
)


In [11]:
# Create Retriever

retriever=vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':1}
)

# Query

Query="Basic Data Types & Control Flow"

results=retriever.invoke(Query)

print(f"Query: {Query}\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}\n")
    print("=" * 80)


Query: Basic Data Types & Control Flow

1. 
As Per Latest
CBSE
Syllabus

Computer Science
Class XI















Flow of control in Python



Flow of Control:


Flow of Control refers to the order in which statements are executed in a program.

Types of Flow:
1. Sequential Flow - one statement after another (default)
2. Conditional Flow - decision-making using if statements
3. Iterative Flow - repeating a set of instructions using loops




Indentation in Python :



Python uses indentation to define blocks of code.

Rules:
• Use consistent indentation (typically 4 spaces)
• Required after if, for, while, etc.







Sequential Flow :


In Sequential Flow, statements are executed line-by-line in the order they appear.

# Example x = 5
y = 10
sum = x + y print("Sum:", sum)



 All statements are executed once in order.



Sequential Flow Conditional flow :

Conditional flow uses decision-making statements to control execution
1. if
2. if-else
3. if-elif-else
1. The if Statement in Pyth

In [17]:
# MMR Retriever

mmr_retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k": 5,
        "lambda_mult":0.8
    }
)

Query1="what is the main difference between a list and a tuple?"
mmr_results=mmr_retriever.invoke(Query1)

print(f"Query: {Query1}\n")

for doc in mmr_results:
    print(f" - {doc.page_content}\n")
    print("=" * 80)

Query: what is the main difference between a list and a tuple?

 - 4. Data Structures in Python 
 
 Prof. Dr. Md. Mijanur Rahman. www.mijanrahman.com 
6 
6 
 
Example 4.2: Sorting List Items Alphanumerically in Python. 
list1 = ['Physics', 'Biology', 'Chemistry', 'Maths', 'History'] 
print ("List before sorting:", list1) 
list1.sort() 
print ("List after sorting: ", list1) 
 
list2 = [10, 40, 30, 50, 20] 
print ("List before sorting:", list2) 
list2.sort() 
print ("List after sorting: ", list2) 
 
Output: 
 
4.3. TUPLE 
 
In Python, a tuple is a built -in data structure that represents an ordered collection of elements. 
Tuples are similar to lists, but they are immutable, meaning their elements cannot be modified 
after creation. Tuples are commonly used for storing heterogen eous data (i.e., data of different 
types) and for representing fixed-size collections of items.  
Following are some examples of tuples in Python: 
my_tuple = (1, 2, 3, 4, 5)    # Tuple of Numbers 
fruits_tuple 

In [20]:
# Configuring LLm Model

from langchain_google_genai import ChatGoogleGenerativeAI

LLM=ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0,
    max_tokens=2000
)

test_response=LLM.invoke("Hello, How are you!")
print(test_response)

content="Hello! I'm doing very well, thank you for asking.\n\nHow can I help you today?" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--9118e5ec-9bfc-4707-ba0b-99e74b8da902-0' usage_metadata={'input_tokens': 7, 'output_tokens': 955, 'total_tokens': 962, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 933}}


In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

system_prompt = (
    "You are a helpful assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer based on the context, say that you don't know. "
    "Keep the answer concise and accurate.\n\n"
    "Context: {context}\n\n"
    "Question: {question}"
)

# Create the prompt template
prompt = ChatPromptTemplate.from_template(system_prompt)

# print(prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LangChain 1.0+ LCEL (LangChain Expression Language)
# This uses the pipe operator (|) to chain components together
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve docs and format them
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | LLM              # Generate answer with LLM
    | StrOutputParser() # Parse output to string
)

print(rag_chain)

query2 = "what is the main difference between a list and a tuple?"

print(f"Query: {query2}")
print("\nProcessing...\n")

# With LangChain 1.0+, we invoke the chain with the question directly
answer = rag_chain.invoke(query2)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)


first={
  context: VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x16075d5e0>, search_kwargs={'k': 1})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are a helpful assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer based on the context, say that you don't know. Keep the answer concise and accurate.\n\nContext: {context}\n\nQuestion: {question}"), additional_kwargs={})]), ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'v

In [30]:
# Wikipedia Retriever

from langchain_community.retrievers import WikipediaRetriever

wikipedia_retriever=WikipediaRetriever(
    top_k_results=2,
    doc_content_chars_max=1000
)

# Query

Query3="What is the history of Python programming language?"

results=wikipedia_retriever.invoke(Query3)

print(f"Query: {Query3}\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}\n")
    print("=" * 80)

Query: What is the history of Python programming language?

1. Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. Currently only versions in the 3.x series are supported. 
Python has gained widespread use in the machine learning community. It is widely taught as an introductory programming language. Since 2003, Python has consistently ranked in the top ten 

In [37]:
Query4="What is the history of Python programming language?"

def hybrid_retriever(query: str) -> str:
    """
    Retrieves information from both local vector store and Wikipedia.

    Args:
        query: The search query

    Returns:
        Formatted string with context from both sources
    """
    
    
    # Get results from vector store
    local_docs = retriever.invoke(Query4)

    # Get results from Wikipedia
    wiki_docs = wikipedia_retriever.invoke(Query4)

    # Combine and format
    context_parts = []

    if local_docs:
        
        # Add local docs content
        for i, doc in enumerate(local_docs, 1):
            print(f"{i}. {doc.page_content}\n")
            print("=" * 80)
        context_parts.append(doc.page_content)
        

    if wiki_docs:
        
        # Add wiki docs content
        for i, doc in enumerate(wiki_docs, 1):
            print(f"{i}. {doc.page_content}\n")
            print("=" * 80)
        context_parts.append(doc.page_content)

    return "\n\n".join(context_parts)

hybrid_retriever(Query4)

1. xxxii    Introduction
One of the most important reasons I continue to use Python is 
because of the Python community, which includes an incredibly diverse 
and welcoming group of people. Community is essential to program -
mers because programming isn’t a solitary pursuit. Most of us, even the 
most experienced programmers, need to ask advice from others who have 
already solved similar problems. Having a well-connected and supportive 
community is critical in helping you solve problems, and the Python com -
munity is fully supportive of people like you who are learning Python as 
your first programming language.
Python is a great language to learn, so let’s get started!

1. Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-ori

"xxxii    Introduction\nOne of the most important reasons I continue to use Python is \nbecause of the Python community, which includes an incredibly diverse \nand welcoming group of people. Community is essential to program -\nmers because programming isn’t a solitary pursuit. Most of us, even the \nmost experienced programmers, need to ask advice from others who have \nalready solved similar problems. Having a well-connected and supportive \ncommunity is critical in helping you solve problems, and the Python com -\nmunity is fully supportive of people like you who are learning Python as \nyour first programming language.\nPython is a great language to learn, so let’s get started!\n\nThe programming language Python was conceived in the late 1980s, and its implementation was started in December 1989 by Guido van Rossum at CWI in the Netherlands as a successor to ABC capable of exception handling and interfacing with the Amoeba operating system. Van Rossum was Python's principal author 